# Section 0: The Fundamentals
### Python fluency — the stuff underneath everything else

This section is different from the others. Each topic has a **short explanation** followed by a 
**small exercise**. The explanations cover the things that quietly trip people up — the loop `else`,
truthiness, aliasing, unpacking. Read the explanation, then do the exercise *without* re-reading it.

If an exercise takes you more than a minute, the concept isn't solid yet. Redo it tomorrow.

**No imports. No NumPy.**

---

## 0.1 — `range` in full

`range` has three forms:

```python
range(stop)              # 0, 1, ..., stop-1
range(start, stop)       # start, start+1, ..., stop-1
range(start, stop, step) # start, start+step, ... (stops before stop)
```

The `stop` is always **exclusive**. The `step` can be negative to count down:

```python
range(5)          # 0 1 2 3 4
range(2, 6)       # 2 3 4 5
range(0, 10, 2)   # 0 2 4 6 8
range(5, 0, -1)   # 5 4 3 2 1
```

A subtle but important fact: `range(...)` is evaluated **once**, when the loop starts. If you change 
the list you're looping over inside the loop, the range doesn't notice. (This is exactly why `replace_pair` 
needed a `while` loop, not a `for`.)

**Exercise 0.1** — Return a list counting down from `n` to `1` (inclusive), then `0` is *not* included.

```
countdown(5) → [5, 4, 3, 2, 1]
```

In [1]:
def countdown(n):
    return [x for x in range(n, 0, -1)]

assert countdown(5) == [5, 4, 3, 2, 1]
assert countdown(1) == [1]
assert countdown(0) == []
print("0.1 passed")

0.1 passed


## 0.2 — `break` and `continue`

**`break`** exits the loop immediately. Nothing after it runs, the loop is over.

```python
for i in range(10):
    if i == 3:
        break
    print(i)   # prints 0 1 2
```

**`continue`** skips the rest of *this* iteration and jumps to the next.

```python
for i in range(5):
    if i % 2 == 0:
        continue
    print(i)   # prints 1 3 (skips evens)
```

The mental model: `break` leaves the building; `continue` skips to the next floor.

**Exercise 0.2** — Sum the numbers in a list, but **skip negatives** and **stop entirely** if you hit a zero.

```
running_sum([1, 2, -5, 3]) → 6      (skipped -5)
running_sum([1, 2, 0, 100]) → 3     (stopped at 0)
```

In [3]:
def running_sum(nums):
    sum = 0

    for num in nums:
        if num < 0:
            continue
        elif num == 0:
            break

        sum += num

    return sum

assert running_sum([1, 2, -5, 3]) == 6
assert running_sum([1, 2, 0, 100]) == 3
assert running_sum([5, 5, 5]) == 15
assert running_sum([]) == 0
print("0.2 passed")

0.2 passed


## 0.3 — The loop `else` (the one that surprised you)

A `for` or `while` loop can have an `else` block. The rule:

> **The `else` runs only if the loop finished *without* hitting a `break`.**

If `break` fired, `else` is skipped. Think of it as "the loop ran to completion, no early exit."

```python
for x in [1, 2, 3]:
    if x == 5:
        break
else:
    print("never found 5")   # this runs — no break happened
```

```python
for x in [1, 2, 3]:
    if x == 2:
        break
else:
    print("never found 2")   # this does NOT run — break fired
```

The canonical use is **search**: loop looking for something, `break` when found, and the `else` 
handles the "searched everything, didn't find it" case — without needing a separate `found = False` flag.

Important: an `else` on a loop that has **no `break`** anywhere is pointless — it always runs. 
(That's why I flagged it in your `interleave`.)

**Exercise 0.3** — Return `True` if `n` is prime, `False` otherwise. Use a loop with `break` and `else`.
(A number is prime if it's ≥ 2 and has no divisor between 2 and n-1.)

```
is_prime(7) → True
is_prime(9) → False
```

In [9]:
def is_prime(n):
    if n < 2:
        return False

    for i in range(2, int(n**0.5)+1):
         if n%i == 0:
             return False

    return True

assert is_prime(2) == True
assert is_prime(7) == True
assert is_prime(9) == False
assert is_prime(1) == False
assert is_prime(13) == True
assert is_prime(15) == False
print("0.3 passed")

0.3 passed


## 0.4 — `enumerate`

When you need both the index and the value, don't do `for i in range(len(lst))` and then `lst[i]`.
Use `enumerate`:

```python
for i, value in enumerate(["a", "b", "c"]):
    print(i, value)
# 0 a
# 1 b
# 2 c
```

You can set a start index: `enumerate(lst, start=1)` begins counting at 1.

This is cleaner and more Pythonic than indexing manually — and it's how you'd build a `char_to_id` 
mapping: `{ch: i for i, ch in enumerate(chars)}`.

**Exercise 0.4** — Return a list of `(index, value)` tuples, but only for values greater than 10.

```
big_with_index([5, 20, 8, 15]) → [(1, 20), (3, 15)]
```

In [11]:
def big_with_index(nums):
    lst = []

    for i, num in enumerate(nums):
        if num > 10:
            lst.append((i, num))

    return lst

assert big_with_index([5, 20, 8, 15]) == [(1, 20), (3, 15)]
assert big_with_index([1, 2, 3]) == []
assert big_with_index([100]) == [(0, 100)]
print("0.4 passed")

0.4 passed


## 0.5 — `zip`

`zip` pairs up elements from multiple iterables, position by position:

```python
for a, b in zip([1, 2, 3], ["x", "y", "z"]):
    print(a, b)
# 1 x
# 2 y
# 3 z
```

It stops at the **shortest** iterable:

```python
list(zip([1, 2, 3], ["a", "b"]))   # [(1, "a"), (2, "b")]
```

Notice: `zip` does the "pair up to the shorter length" part of your `interleave` problem automatically.
Your adjacent-pairs problem (1.2) can also be written as `zip(lst, lst[1:])`.

**Exercise 0.5** — Given two lists of equal length (names and scores), return a dict mapping name → score.

```
make_scores(["ann", "bob"], [90, 75]) → {"ann": 90, "bob": 75}
```

**BONUS:** rewrite `adjacent_pairs` from 1.2 using `zip` in one line.

In [12]:
def make_scores(names, scores):
    return dict(zip(names, scores))

assert make_scores(["ann", "bob"], [90, 75]) == {"ann": 90, "bob": 75}
assert make_scores([], []) == {}
assert make_scores(["x"], [1]) == {"x": 1}
print("0.5 passed")

0.5 passed


## 0.6 — Truthiness

In a boolean context (`if`, `while`), Python treats some values as `False` even though they aren't 
literally `False`. The **falsy** values:

```python
False, None, 0, 0.0, "", [], {}, set(), ()
```

Everything else is **truthy** — including non-empty strings, non-empty lists, non-zero numbers.

This means you write:

```python
if my_list:          # "if the list is non-empty"
if not my_string:    # "if the string is empty"
while remaining:     # "while there's anything left"
```

instead of `if len(my_list) > 0:`. Cleaner, and idiomatic.

One trap: `0` is falsy. So `if x:` when `x` could legitimately be `0` is a bug waiting to happen — 
use `if x is not None:` when you specifically mean a value was provided.

**Exercise 0.6** — Given a list of values (could contain numbers, strings, empty containers), 
return only the **truthy** ones.

```
keep_truthy([0, 1, "", "hi", [], [1], None, 5]) → [1, "hi", [1], 5]
```

In [14]:
def keep_truthy(values):
    true_values = []

    for val in values:
        if val:
            true_values.append(val)

    return true_values

assert keep_truthy([0, 1, "", "hi", [], [1], None, 5]) == [1, "hi", [1], 5]
assert keep_truthy([0, "", None]) == []
assert keep_truthy([1, 2, 3]) == [1, 2, 3]
print("0.6 passed")

0.6 passed


## 0.7 — The conditional expression (ternary)

A compact `if`/`else` that produces a value:

```python
result = "even" if n % 2 == 0 else "odd"
```

Reads as: *value-if-true* `if` *condition* `else` *value-if-false*.

Use it for simple either/or value selection. Don't nest them — if you need more than one, use a 
normal `if` block. It's for clarity, not for cramming logic onto one line.

**Exercise 0.7** — Return the absolute difference between two numbers, using a ternary 
(don't use `abs`).

```
abs_diff(3, 8) → 5
abs_diff(8, 3) → 5
```

In [15]:
def abs_diff(a, b):
    return a-b if a > b else b-a

assert abs_diff(3, 8) == 5
assert abs_diff(8, 3) == 5
assert abs_diff(4, 4) == 0
print("0.7 passed")

0.7 passed


## 0.8 — Unpacking

You can assign multiple variables from a sequence in one line:

```python
a, b = (1, 2)        # a=1, b=2
x, y, z = [10, 20, 30]
```

Swap without a temp variable:

```python
a, b = b, a
```

Star-unpacking grabs "the rest" into a list:

```python
first, *rest = [1, 2, 3, 4]    # first=1, rest=[2, 3, 4]
*init, last = [1, 2, 3, 4]     # init=[1, 2, 3], last=4
```

This is why `for i, value in enumerate(...)` and `for a, b in zip(...)` work — each item is a 
tuple being unpacked. And when you wrote `pair = (lst[i], lst[i+1])` then checked `pair[0]`, 
`pair[1]` — you could unpack instead: `a, b = pair`.

**Exercise 0.8** — Given a list with at least two elements, return a tuple of 
`(first_element, last_element, everything_in_between_as_a_list)`.

```
ends_and_middle([1, 2, 3, 4, 5]) → (1, 5, [2, 3, 4])
ends_and_middle([1, 2]) → (1, 2, [])
```

Use star-unpacking.

In [21]:
def ends_and_middle(lst):
    first, *rest, last = lst
    return (first, last, rest)

assert ends_and_middle([1, 2, 3, 4, 5]) == (1, 5, [2, 3, 4])
assert ends_and_middle([1, 2]) == (1, 2, [])
assert ends_and_middle([1, 2, 3]) == (1, 3, [2])
print("0.8 passed")

0.8 passed


## 0.9 — `is` vs `==`, and `None`

`==` asks: **are these equal in value?**  
`is` asks: **are these the exact same object in memory?**

```python
[1, 2] == [1, 2]   # True  — same contents
[1, 2] is [1, 2]   # False — two different list objects
```

For almost everything, you want `==`. The one place you use `is`: comparing to `None`, `True`, `False` — 
the singletons. The idiom is:

```python
if x is None:
    ...
if x is not None:
    ...
```

Why not `== None`? It works, but `is None` is the convention and slightly safer (a custom object 
could override `==` to lie about equality; nothing can fake object identity).

**Exercise 0.9** — Given a list that may contain `None` values mixed with numbers, 
return the sum of just the numbers.

```
sum_skip_none([1, None, 3, None, 5]) → 9
```

In [24]:
def sum_skip_none(values):
    total = 0

    for val in values:
        if val is not None:
            total += val

    return total

assert sum_skip_none([1, None, 3, None, 5]) == 9
assert sum_skip_none([None, None]) == 0
assert sum_skip_none([10]) == 10
assert sum_skip_none([0, None]) == 0   # careful: 0 is a real value, not None
print("0.9 passed")

0.9 passed


## 0.10 — Aliasing and mutation (the `.copy()` lesson)

When you assign a list to a new variable, you **don't** get a copy. You get a second name pointing 
at the *same* list:

```python
a = [1, 2, 3]
b = a          # b and a are the SAME list
b.append(4)
print(a)       # [1, 2, 3, 4]  — a changed too!
```

To get an independent copy:

```python
b = a.copy()       # or list(a), or a[:]
b.append(4)
print(a)           # [1, 2, 3] — unchanged
```

This is exactly the issue from `replace_pair`: mutating the argument changed the caller's list. 
Making a copy first (`replaced = ids.copy()`) fixed it.

The distinction: **reassignment** (`x = something`) rebinds the name. **Mutation** 
(`.append`, `.pop`, `x[i] = ...`) changes the object in place, and every name pointing at it sees the change.

**Exercise 0.10** — Write a function that takes a list and returns a **new** list with the 
last element removed. The original must be unchanged.

```
without_last([1, 2, 3]) → [1, 2]   and the input list still equals [1, 2, 3] afterward
```

In [26]:
def without_last(lst):
    return lst.copy()[:-1]

original = [1, 2, 3]
assert without_last(original) == [1, 2]
assert original == [1, 2, 3], "you mutated the input!"
assert without_last([5]) == []
print("0.10 passed")

0.10 passed


## 0.11 — Iterating over dictionaries

Three ways to loop over a dict:

```python
d = {"a": 1, "b": 2}

for key in d:              # iterates over KEYS by default
    print(key)             # a, b

for key in d.keys():       # same thing, explicit
    ...

for value in d.values():   # iterates over VALUES
    print(value)           # 1, 2

for key, value in d.items():   # both at once (unpacking again!)
    print(key, value)          # a 1, b 2
```

And `.get()` for safe lookup with a default when the key might be missing:

```python
d.get("a")        # 1
d.get("z")        # None  (no error)
d.get("z", 0)     # 0     (your chosen default)
```

`.get(key, default)` is the clean way to avoid `KeyError` — useful when counting 
(`counts[k] = counts.get(k, 0) + 1`).

**Exercise 0.11** — Given a dict of word→count, return the total count of all words 
whose key starts with a vowel (a, e, i, o, u).

```
vowel_total({"apple": 3, "bee": 2, "egg": 1, "dog": 5}) → 4   (apple:3 + egg:1)
```

In [28]:
def vowel_total(counts):
    count = 0

    for word, val in counts.items():
        if word[0] in "aeiou":
            count += val

    return count

assert vowel_total({"apple": 3, "bee": 2, "egg": 1, "dog": 5}) == 4
assert vowel_total({"dog": 5}) == 0
assert vowel_total({}) == 0
assert vowel_total({"ant": 1, "owl": 2}) == 3
print("0.11 passed")

0.11 passed


## 0.12 — Essential string methods

The ones you'll reach for constantly:

```python
"a,b,c".split(",")        # ["a", "b", "c"]
"hello world".split()     # ["hello", "world"]  (splits on whitespace)
"-".join(["a", "b", "c"]) # "a-b-c"
"  hi  ".strip()          # "hi"  (removes leading/trailing whitespace)
"hello".replace("l", "L") # "heLLo"  (all occurrences)
"hello".startswith("he")  # True
"hello".upper()           # "HELLO"
"HELLO".lower()           # "hello"
```

`split` and `join` are inverses. `join` is a method **on the separator string** — that trips 
people up: it's `separator.join(list)`, not `list.join(separator)`.

You'll use these everywhere in tokenization and the corpus prep you were doing earlier.

**Exercise 0.12** — Given a sentence, return it with the word order reversed.

```
reverse_words("the cat sat") → "sat cat the"
```

Use `split` and `join`.

In [29]:
def reverse_words(sentence):
    split = sentence.split()
    return " ".join(reversed(split))

assert reverse_words("the cat sat") == "sat cat the"
assert reverse_words("hello") == "hello"
assert reverse_words("a b c d") == "d c b a"
print("0.12 passed")

0.12 passed


---

## Done with Section 0?

These twelve are the bedrock. Loop control, truthiness, unpacking, aliasing, dict iteration, string 
surgery — they show up in every single thing you'll build this summer. When they're automatic, you 
stop thinking about the language and start thinking about the problem.

When you can do all twelve without hesitation, the Section 1–8 problems (and BPE) will feel like a 
different experience. Then go start Day 5.